In [ ]:
import os
import zipfile
import pyarrow.parquet as pq
import pandas as pd

folder = r"F:\sanazi\python\data science\code\tamrin\proje\Tennis Schema\Tennis Schema\tennis_data"

data = {}

for file in os.listdir(folder): 
    if file.endswith(".zip"):

        zip_path = os.path.join(folder, file) 

        with zipfile.ZipFile(zip_path) as z: 
            for parquet_file in z.namelist(): 
                if parquet_file.endswith(".parquet"): 

                    with z.open(parquet_file) as f:
                        df = pq.read_table(f).to_pandas()
                    table_name =os.path.basename(os.path.dirname(parquet_file))[4:-8] + os.path.basename(parquet_file)[:-17]

                    if table_name not in data:
                        data[table_name] = [] 
                    data[table_name].append(df)

for table_name in data:
    data[table_name] = pd.concat(data[table_name], ignore_index=True) 

print(data.keys())

In [ ]:
import pandas as pd
import numpy as np


event = data["matchevent"].copy()


home = data["matchhome_team"][["match_id", "full_name", "weight", "gender"]].rename(
    columns={
        "full_name": "home_player",
        "weight": "home_weight",
        "gender": "home_gender"
    }
)


away = data["matchaway_team"][["match_id", "full_name", "weight", "gender"]].rename(
    columns={
        "full_name": "away_player",
        "weight": "away_weight",
        "gender": "away_gender"
    }
)


df = (
    event.merge(home, on="match_id")
         .merge(away, on="match_id")
)


df["winner_weight"] = np.where(df["winner_code"] == 1,
                               df["home_weight"],
                               df["away_weight"])

df["loser_weight"] = np.where(df["winner_code"] == 1,
                              df["away_weight"],
                              df["home_weight"])

df["gender"] = np.where(df["winner_code"] == 1,
                        df["home_gender"],
                        df["away_gender"])


df = df.dropna(subset=["winner_weight", "loser_weight", "gender"])

result = (
    df.groupby("gender")[["winner_weight", "loser_weight"]]
      .mean()
      .round(2)
)

print(result)

In [ ]:
import matplotlib.pyplot as plt

result.plot(kind="bar")

plt.title("Average Weight of Winners and Losers by Gender")
plt.xlabel("Gender")
plt.ylabel("Average Weight (kg)")
plt.xticks(rotation=0)

plt.show()